# Notebook 04 - Budget Reallocation Simulation (RQ3)

**RQ3:** How much does reallocating the marketing budget based on attribution results improve simulated conversion outcomes?

This notebook uses CSV outputs from RQ1.  The simulation is comparative, not a real revenue forecast.

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd().resolve(), Path.cwd().resolve() / "notebooks", Path.cwd().resolve() / "analysis_python" / "notebooks", Path.cwd().resolve().parent / "notebooks"]:
    if (candidate / "notebook_header.py").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("Could not find notebook_header.py")

import numpy as np
import pandas as pd

from notebook_header import OUTPUT_DIR as BASE_OUTPUT_DIR, RANDOM_SEED, load_touchpoints

OUTPUT_DIR = BASE_OUTPUT_DIR / "rq3_simulation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)
np.random.seed(RANDOM_SEED)

## 1. Simulation Assumptions

The dataset does not contain actual spend or revenue. The following assumptions are used only to compare scenarios on a common scale.

In [2]:
REVENUE_PER_CONVERSION = 100.0
TOTAL_BUDGET = 100_000.0
TOTAL_TOUCHPOINTS = 10_000
COST_PER_TOUCH = TOTAL_BUDGET / TOTAL_TOUCHPOINTS
SENSITIVITY_FACTORS = [0.8, 1.0, 1.2]

assumptions = pd.DataFrame([
    {"parameter": "revenue_per_conversion", "value": REVENUE_PER_CONVERSION},
    {"parameter": "total_budget", "value": TOTAL_BUDGET},
    {"parameter": "total_touchpoints", "value": TOTAL_TOUCHPOINTS},
    {"parameter": "cost_per_touch", "value": COST_PER_TOUCH},
    {"parameter": "sensitivity_factors", "value": str(SENSITIVITY_FACTORS)},
    {"parameter": "random_seed", "value": RANDOM_SEED},
])

assumptions

,parameter,value
0,revenue_per_conversion,100.0
1,total_budget,100000.0
2,total_touchpoints,10000
3,cost_per_touch,10.0
4,sensitivity_factors,"[0.8, 1.0, 1.2]"
5,random_seed,42


## 2. Load RQ1 Outputs

In [3]:
df_tp = load_touchpoints()
attribution = pd.read_csv(OUTPUT_DIR.parent / "rq1_basic" / "01_attribution_share.csv", index_col=0)
channel_conversion = pd.read_csv(OUTPUT_DIR.parent / "rq1_basic" / "01_channel_conversion_rate.csv", index_col=0)

channels = sorted(attribution.index.tolist())
if "conversion_rate" in channel_conversion.columns:
    conversion_rate = channel_conversion["conversion_rate"].reindex(channels)
else:
    conversion_rate = (channel_conversion["conversion_rate_pct"] / 100).reindex(channels)
linear_share = (attribution["linear_pct"] / attribution["linear_pct"].sum()).reindex(channels)

assert len(df_tp) == TOTAL_TOUCHPOINTS
assert len(channels) == 6
assert conversion_rate.notna().all()
assert abs(linear_share.sum() - 1) < 1e-9

pd.DataFrame({"conversion_rate": conversion_rate, "linear_budget_weight": linear_share})

,conversion_rate,linear_budget_weight
Direct Traffic,0.4956,0.1715
Display Ads,0.4961,0.1710
Email,0.5018,0.1628
Referral,0.4991,0.1675
Search Ads,0.4798,0.1602
Social Media,0.4934,0.1670


## 3. Define Budget Scenarios

- **S0_equal:** equal budget split across six channels.
- **S1_conversion_rate:** budget weighted by row-level conversion rate.
- **S2_linear_attribution:** budget weighted by Linear attribution share from RQ1.

In [4]:
w_s0 = pd.Series(1 / len(channels), index=channels, name="S0_equal")
w_s1 = (conversion_rate / conversion_rate.sum()).rename("S1_conversion_rate")
w_s2 = linear_share.rename("S2_linear_attribution")

budget_weights = pd.concat([w_s0, w_s1, w_s2], axis=1)

for col in budget_weights.columns:
    assert abs(budget_weights[col].sum() - 1) < 1e-9

budget_weights.round(6)

,S0_equal,S1_conversion_rate,S2_linear_attribution
Direct Traffic,0.1667,0.1671,0.1715
Display Ads,0.1667,0.1673,0.1710
Email,0.1667,0.1692,0.1628
Referral,0.1667,0.1683,0.1675
Search Ads,0.1667,0.1618,0.1602
Social Media,0.1667,0.1664,0.1670


## 4. Simulate Conversions, Revenue, and ROI

In [5]:
def simulate(weights, revenue_per_conversion=REVENUE_PER_CONVERSION):
    spend = TOTAL_BUDGET * weights
    simulated_touchpoints = spend / COST_PER_TOUCH
    conversions = simulated_touchpoints * conversion_rate.reindex(weights.index)
    revenue = conversions * revenue_per_conversion
    per_channel = pd.DataFrame({
        "budget_weight": weights,
        "spend": spend,
        "simulated_touchpoints": simulated_touchpoints,
        "conversion_rate": conversion_rate.reindex(weights.index),
        "simulated_conversions": conversions,
        "simulated_revenue": revenue,
        "roi": revenue / spend - 1,
    })
    return float(conversions.sum()), float(revenue.sum()), per_channel


scenario_map = {
    "S0_equal": w_s0,
    "S1_conversion_rate": w_s1,
    "S2_linear_attribution": w_s2,
}

scenario_results = {}
for scenario, weights in scenario_map.items():
    conversions, revenue, per_channel = simulate(weights)
    scenario_results[scenario] = {
        "conversions": conversions,
        "revenue": revenue,
        "per_channel": per_channel,
    }

baseline_conversions = scenario_results["S0_equal"]["conversions"]
baseline_revenue = scenario_results["S0_equal"]["revenue"]

simulation_results = pd.DataFrame([
    {
        "scenario": scenario,
        "conversions": values["conversions"],
        "revenue": values["revenue"],
        "delta_conversions": values["conversions"] - baseline_conversions,
        "delta_revenue": values["revenue"] - baseline_revenue,
        "delta_conv_pct": (values["conversions"] / baseline_conversions - 1) * 100,
        "delta_rev_pct": (values["revenue"] / baseline_revenue - 1) * 100,
    }
    for scenario, values in scenario_results.items()
])

simulation_results.round(4)

,scenario,conversions,revenue,delta_conversions,delta_revenue,delta_conv_pct,delta_rev_pct
0,S0_equal,4943.0895,494308.9475,0.0000,0.0000,0.0000,0.0000
1,S1_conversion_rate,4944.0865,494408.6532,0.9971,99.7057,0.0202,0.0202
2,S2_linear_attribution,4943.9167,494391.6659,0.8272,82.7184,0.0167,0.0167


## 5. Sensitivity Analysis

In [6]:
sensitivity_rows = []
for factor in SENSITIVITY_FACTORS:
    revenue_per_conversion = REVENUE_PER_CONVERSION * factor
    _, baseline_revenue_factor, _ = simulate(w_s0, revenue_per_conversion=revenue_per_conversion)
    for scenario, weights in scenario_map.items():
        conversions, revenue, _ = simulate(weights, revenue_per_conversion=revenue_per_conversion)
        sensitivity_rows.append({
            "scenario": scenario,
            "revenue_factor": factor,
            "revenue_per_conversion": revenue_per_conversion,
            "conversions": conversions,
            "revenue": revenue,
            "delta_rev_pct_vs_S0": (revenue / baseline_revenue_factor - 1) * 100,
        })

simulation_sensitivity = pd.DataFrame(sensitivity_rows)
simulation_sensitivity.round(4)

,scenario,revenue_factor,revenue_per_conversion,conversions,revenue,delta_rev_pct_vs_S0
0,S0_equal,0.8,80.0,4943.0895,395447.1580,0.0000
1,S1_conversion_rate,0.8,80.0,4944.0865,395526.9226,0.0202
2,S2_linear_attribution,0.8,80.0,4943.9167,395513.3327,0.0167
3,S0_equal,1.0,100.0,4943.0895,494308.9475,0.0000
4,S1_conversion_rate,1.0,100.0,4944.0865,494408.6532,0.0202
5,S2_linear_attribution,1.0,100.0,4943.9167,494391.6659,0.0167
6,S0_equal,1.2,120.0,4943.0895,593170.7370,0.0000
7,S1_conversion_rate,1.2,120.0,4944.0865,593290.3839,0.0202
8,S2_linear_attribution,1.2,120.0,4943.9167,593269.9990,0.0167


## 6. Save Outputs

In [7]:
roi_by_channel = pd.DataFrame({
    scenario: values["per_channel"]["roi"] for scenario, values in scenario_results.items()
})

budget_weights.to_csv(OUTPUT_DIR / "04_budget_weights.csv")
simulation_results.to_csv(OUTPUT_DIR / "04_results.csv", index=False)
roi_by_channel.to_csv(OUTPUT_DIR / "04_roi_by_channel.csv")
simulation_sensitivity.to_csv(OUTPUT_DIR / "04_sensitivity.csv", index=False)
assumptions.to_csv(OUTPUT_DIR / "04_assumptions.csv", index=False)

for scenario, values in scenario_results.items():
    values["per_channel"].to_csv(OUTPUT_DIR / f"04_per_channel_{scenario}.csv")

print("Saved RQ3 simulation outputs to outputs/rq3_simulation/.")

Saved RQ3 simulation outputs to outputs/rq3_simulation/.


## 7. RQ3 conclusion

RQ3 shows only a very small simulated gain from budget reallocation. Under the equal-budget baseline, the model estimates 4,943.09 conversions and $494,308.95 in simulated revenue. Weighting the budget by row-level conversion rates increases the estimate to 4,944.09 conversions and $494,408.65 in simulated revenue, a gain of 0.997 conversions, $99.71, or 0.0202%. Weighting by Linear attribution gives 4,943.92 conversions and $494,391.67 in simulated revenue, a gain of 0.827 conversions, $82.72, or 0.0167%. In practical terms, the difference between equal allocation and attribution-based allocation is almost negligible.

This result follows directly from the earlier findings. RQ1 showed that channel conversion rates are tightly clustered between 47.98% and 50.18%, and RQ2 showed that channel identity is a weaker predictor than journey length. With so little separation between channels, shifting budget from one to another has limited room to move the simulated outcome in any meaningful direction.

The sensitivity analysis, which varied the assumed revenue per conversion by ±20% from the $100 baseline, uses $80, $100, and $120 per conversion. This changes the absolute revenue figures across all scenarios but leaves the ranking between them unchanged. This is unsurprising: since the same revenue-per-conversion value is applied across all three scenarios, any scaling affects them proportionally and does not alter which scenario performs best relative to the others.

What RQ3 offers, then, is primarily methodological. The simulation demonstrates how attribution weights can be translated into estimated conversions and revenue under clearly stated assumptions, a framework that could produce more actionable results in a dataset where channels are more differentiated. In this particular case, the channel mix is balanced enough that the simulated uplift is too small to justify any strong reallocation recommendation. Since revenue is calculated using a uniform assumed value of $100 per conversion, these figures are best understood as a controlled illustration rather than a forecast of real-world ROI.
